# SQL 窗口函数 (Window Functions)

> **适用场景**: 高级数据分析、排名、滑动计算、同环比分析
> **面试频率**: ⭐⭐⭐⭐⭐ 极高频

## 什么是窗口函数？

窗口函数在**不减少行数**的情况下，对一组相关行（称为`窗口`）执行计算。与 `GROUP BY` 不同，窗口函数**保留每一行的详细信息**，同时附加聚合计算结果。

```
基本语法:
函数名() OVER (
    [PARTITION BY 分组列]
    [ORDER BY 排序列]
    [ROWS/RANGE BETWEEN ... AND ...]
)
```

---

## 目录
1. ROW_NUMBER / RANK / DENSE_RANK 区别
2. LAG / LEAD 取前后行
3. PARTITION BY + ORDER BY 组合
4. ROWS BETWEEN / RANGE BETWEEN 滑动窗口
5. FIRST_VALUE / LAST_VALUE / NTH_VALUE
6. NTILE 分桶
7. 练习题

In [3]:
# 安装并导入 DuckDB
# !pip install duckdb -q
import duckdb
import pandas as pd

# 创建内存数据库连接
con = duckdb.connect()

print(f"DuckDB version: {duckdb.__version__}")
print("连接成功！")

DuckDB version: 1.5.2
连接成功！


In [ ]:
# 创建示例数据：电商销售数据
con.execute("""
CREATE OR REPLACE TABLE sales AS
SELECT * FROM (VALUES
    (1,  'Alice',   'Electronics', 1200.00, DATE '2024-01-15'),
    (2,  'Bob',     'Clothing',     350.00, DATE '2024-01-16'),
    (3,  'Alice',   'Electronics',  800.00, DATE '2024-01-20'),
    (4,  'Charlie', 'Electronics',  950.00, DATE '2024-01-22'),
    (5,  'Bob',     'Electronics',  430.00, DATE '2024-01-25'),
    (6,  'Alice',   'Clothing',     200.00, DATE '2024-02-01'),
    (7,  'Charlie', 'Clothing',     150.00, DATE '2024-02-05'),
    (8,  'Bob',     'Clothing',     520.00, DATE '2024-02-10'),
    (9,  'Alice',   'Electronics', 2200.00, DATE '2024-02-15'),
    (10, 'Charlie', 'Clothing',     890.00, DATE '2024-02-20'),
    (11, 'Bob',     'Electronics',  670.00, DATE '2024-02-22'),
    (12, 'Alice',   'Clothing',     310.00, DATE '2024-03-01')
) t(sale_id, salesperson, category, amount, sale_date)
""")

# 创建员工薪资数据
con.execute("""
CREATE OR REPLACE TABLE employees AS
SELECT * FROM (VALUES
    (1,  'Alice',   'Engineering',  95000, '2020-03-01'),
    (2,  'Bob',     'Engineering',  88000, '2021-06-15'),
    (3,  'Carol',   'Engineering', 102000, '2019-01-10'),
    (4,  'David',   'Marketing',    72000, '2022-08-20'),
    (5,  'Eve',     'Marketing',    68000, '2021-11-30'),
    (6,  'Frank',   'Marketing',    75000, '2020-05-14'),
    (7,  'Grace',   'HR',           58000, '2023-01-05'),
    (8,  'Heidi',   'HR',           62000, '2022-03-22'),
    (9,  'Ivan',    'Engineering',  91000, '2021-09-01'),
    (10, 'Judy',    'Marketing',    79000, '2019-07-18')
) t(emp_id, name, department, salary, hire_date)
""")

print("示例数据创建完成！")
print("\n--- sales 表 ---")
con.execute("SELECT * FROM sales ORDER BY sale_id").df()

---

## 1. ROW_NUMBER / RANK / DENSE_RANK 区别

这三个函数都用于排名，但处理**并列（tie）**时行为不同：

| 函数 | 并列时 | 跳过编号 | 典型场景 |
|------|--------|----------|----------|
| `ROW_NUMBER()` | 强制唯一，任意顺序分配 | 否 | 去重、取第N行 |
| `RANK()` | 并列相同名次，后续跳号 | 是 (1,1,3) | 奖牌榜，允许空缺 |
| `DENSE_RANK()` | 并列相同名次，后续不跳号 | 否 (1,1,2) | 薪资分级，不允许空缺 |

**面试考点**: 经典的 "每组取Top N" 问题通常用 `ROW_NUMBER()` + CTE 实现。

In [ ]:
# 演示三种排名函数的差异
# 场景：对每个部门的员工按薪资排名
result = con.execute("""
SELECT
    name,
    department,
    salary,
    ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS row_num,
    RANK()       OVER (PARTITION BY department ORDER BY salary DESC) AS rnk,
    DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dense_rnk
FROM employees
ORDER BY department, salary DESC
""").df()

print("部门内薪资排名对比（注意 Engineering 部门中没有并列，但可以看到排名逻辑）")
result

In [ ]:
# 构造并列场景来清晰展示差异
con.execute("""
CREATE OR REPLACE TABLE scores AS
SELECT * FROM (VALUES
    ('Alice', 95),
    ('Bob',   88),
    ('Carol', 88),
    ('David', 75),
    ('Eve',   75),
    ('Frank', 70)
) t(name, score)
""")

result = con.execute("""
SELECT
    name,
    score,
    ROW_NUMBER() OVER (ORDER BY score DESC) AS row_num,   -- 1,2,3,4,5,6
    RANK()       OVER (ORDER BY score DESC) AS rnk,       -- 1,2,2,4,4,6  (跳过3和5)
    DENSE_RANK() OVER (ORDER BY score DESC) AS dense_rnk  -- 1,2,2,3,3,4  (不跳过)
FROM scores
ORDER BY score DESC
""").df()

print("并列场景下的排名差异:")
print("ROW_NUMBER: 强制唯一 (1,2,3,4,5,6)")
print("RANK:       并列跳号 (1,2,2,4,4,6)")
print("DENSE_RANK: 并列不跳号 (1,2,2,3,3,4)")
result

In [ ]:
# 经典面试题：每个类别中销售额最高的 Top 2 销售人员
# 方法：ROW_NUMBER() + CTE
result = con.execute("""
WITH ranked_sales AS (
    SELECT
        salesperson,
        category,
        SUM(amount) AS total_amount,
        ROW_NUMBER() OVER (
            PARTITION BY category
            ORDER BY SUM(amount) DESC
        ) AS rn
    FROM sales
    GROUP BY salesperson, category
)
SELECT
    category,
    salesperson,
    total_amount,
    rn AS rank_in_category
FROM ranked_sales
WHERE rn <= 2
ORDER BY category, rn
""").df()

print("每个类别 Top 2 销售人员（经典面试模式）:")
result

---

## 2. LAG / LEAD 取前后行

`LAG(col, n, default)` 取**前 n 行**的值（向后看）
`LEAD(col, n, default)` 取**后 n 行**的值（向前看）

**典型应用场景**:
- 环比计算（本月 vs 上月）
- 同比计算（今年 vs 去年同期）
- 计算连续两行之间的差值
- 找出状态变化（事件检测）

```sql
LAG(column, offset, default_value) OVER (PARTITION BY ... ORDER BY ...)
--          ^^^^^^  ^^^^^^^^^^^^^  -- offset 默认为 1，default 在无前值时返回
```

In [ ]:
# 月度销售额环比分析
result = con.execute("""
WITH monthly_sales AS (
    SELECT
        DATE_TRUNC('month', sale_date) AS month,
        SUM(amount) AS total_amount
    FROM sales
    GROUP BY DATE_TRUNC('month', sale_date)
)
SELECT
    month,
    total_amount,
    LAG(total_amount, 1, 0) OVER (ORDER BY month) AS prev_month_amount,
    LEAD(total_amount, 1, 0) OVER (ORDER BY month) AS next_month_amount,
    -- 环比增长率
    ROUND(
        (total_amount - LAG(total_amount, 1) OVER (ORDER BY month))
        / LAG(total_amount, 1) OVER (ORDER BY month) * 100,
        2
    ) AS mom_growth_pct  -- Month-over-Month
FROM monthly_sales
ORDER BY month
""").df()

print("月度销售额环比分析:")
result

In [ ]:
# 检测销售额变化趋势（上升/下降/持平）
result = con.execute("""
WITH monthly_sales AS (
    SELECT
        salesperson,
        DATE_TRUNC('month', sale_date) AS month,
        SUM(amount) AS total_amount
    FROM sales
    GROUP BY salesperson, DATE_TRUNC('month', sale_date)
)
SELECT
    salesperson,
    month,
    total_amount,
    LAG(total_amount) OVER (PARTITION BY salesperson ORDER BY month) AS prev_amount,
    CASE
        WHEN total_amount > LAG(total_amount) OVER (PARTITION BY salesperson ORDER BY month)
             THEN '↑ 上升'
        WHEN total_amount < LAG(total_amount) OVER (PARTITION BY salesperson ORDER BY month)
             THEN '↓ 下降'
        WHEN LAG(total_amount) OVER (PARTITION BY salesperson ORDER BY month) IS NULL
             THEN '- 首月'
        ELSE '→ 持平'
    END AS trend
FROM monthly_sales
ORDER BY salesperson, month
""").df()

print("按销售员的月度趋势分析:")
result

---

## 3. PARTITION BY + ORDER BY 组合

`PARTITION BY` 将数据分组，类似 GROUP BY，但不折叠行。
`ORDER BY` 在窗口内定义行的顺序，启用累计计算。

**关键理解**:
- 只有 `PARTITION BY`: 整个分区内的聚合（类似 GROUP BY，但保留行）
- 只有 `ORDER BY`: 从第一行到当前行的累计计算
- 两者都有: 分区内从第一行到当前行的累计

**运行总计（Running Total）** 是高频面试模式！

In [ ]:
# 演示不同 OVER() 子句的效果
result = con.execute("""
SELECT
    sale_id,
    salesperson,
    category,
    amount,
    -- 无分区无排序：全表总计
    SUM(amount) OVER ()                                     AS grand_total,
    -- 只有 PARTITION BY：按类别小计
    SUM(amount) OVER (PARTITION BY category)                AS category_total,
    -- 只有 ORDER BY：按日期累计运行总计
    SUM(amount) OVER (ORDER BY sale_date)                   AS running_total,
    -- PARTITION BY + ORDER BY：按销售员分区的累计
    SUM(amount) OVER (
        PARTITION BY salesperson
        ORDER BY sale_date
    )                                                       AS person_running_total
FROM sales
ORDER BY salesperson, sale_date
""").df()

print("不同 OVER() 子句效果对比:")
result

In [ ]:
# 百分比贡献计算（每行相对于分区总计的百分比）
result = con.execute("""
SELECT
    sale_id,
    salesperson,
    category,
    amount,
    -- 个人总销售额
    SUM(amount) OVER (PARTITION BY salesperson) AS person_total,
    -- 该笔交易占个人总销售额的比例
    ROUND(amount / SUM(amount) OVER (PARTITION BY salesperson) * 100, 2) AS pct_of_person,
    -- 该笔交易占全公司总销售额的比例
    ROUND(amount / SUM(amount) OVER () * 100, 2) AS pct_of_total
FROM sales
ORDER BY salesperson, sale_date
""").df()

print("销售额占比分析:")
result

---

## 4. ROWS BETWEEN / RANGE BETWEEN 滑动窗口

窗口帧（Frame）定义了在窗口内**哪些行参与计算**。

### 帧边界关键字
```
UNBOUNDED PRECEDING  -- 分区的第一行
n PRECEDING          -- 当前行前 n 行
CURRENT ROW          -- 当前行
n FOLLOWING          -- 当前行后 n 行
UNBOUNDED FOLLOWING  -- 分区的最后一行
```

### ROWS vs RANGE 的关键区别

| 对比 | ROWS BETWEEN | RANGE BETWEEN |
|------|-------------|---------------|
| 计算基础 | **物理行数** | **逻辑值范围** |
| 并列行处理 | 精确到每一物理行 | 相同 ORDER BY 值的行视为同组 |
| 常用场景 | 固定行数滑动窗口 | 基于时间/值范围的窗口 |

**默认帧**: 当有 ORDER BY 时，默认为 `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`

In [ ]:
# 构造一个清晰展示 ROWS vs RANGE 差异的示例
con.execute("""
CREATE OR REPLACE TABLE daily_sales AS
SELECT * FROM (VALUES
    (DATE '2024-01-01', 100),
    (DATE '2024-01-02', 200),
    (DATE '2024-01-02', 150),  -- 同一天两条记录（并列）
    (DATE '2024-01-03', 300),
    (DATE '2024-01-04', 250),
    (DATE '2024-01-05', 180)
) t(sale_date, amount)
""")

result = con.execute("""
SELECT
    sale_date,
    amount,
    -- ROWS: 精确取当前物理行及之前1行（共2行）
    SUM(amount) OVER (
        ORDER BY sale_date
        ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
    ) AS rows_2day_sum,
    -- RANGE: 取 ORDER BY 值范围内的所有行（同一天的都算当前行）
    SUM(amount) OVER (
        ORDER BY sale_date
        RANGE BETWEEN 1 PRECEDING AND CURRENT ROW  -- 这里 1 PRECEDING 是1天
    ) AS range_2day_sum,
    -- 经典3日滑动平均
    AVG(amount) OVER (
        ORDER BY sale_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS rolling_3day_avg
FROM daily_sales
ORDER BY sale_date, amount
""").df()

print("ROWS vs RANGE 的差异（注意 2024-01-02 的两行）:")
print("ROWS 2day_sum: 严格取物理前1行 + 当前行")
print("RANGE 2day_sum: 取值范围内所有行（同日期全部包含）")
result

In [ ]:
# 实际业务场景：7日滑动平均 & 累计总计
result = con.execute("""
WITH monthly_totals AS (
    SELECT
        DATE_TRUNC('month', sale_date) AS month,
        SUM(amount) AS monthly_amount
    FROM sales
    GROUP BY 1
    ORDER BY 1
)
SELECT
    month,
    monthly_amount,
    -- 累计总计（从头累加）
    SUM(monthly_amount) OVER (
        ORDER BY month
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_total,
    -- 向前2个月 + 当前月的滑动总和（3个月窗口）
    SUM(monthly_amount) OVER (
        ORDER BY month
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS rolling_3month_sum,
    -- 滑动均值
    ROUND(AVG(monthly_amount) OVER (
        ORDER BY month
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS rolling_3month_avg
FROM monthly_totals
""").df()

print("月度销售额滑动窗口分析:")
result

---

## 5. FIRST_VALUE / LAST_VALUE / NTH_VALUE

这些函数从窗口帧中取特定位置的值：

- `FIRST_VALUE(col)`: 取窗口帧内**第一行**的值
- `LAST_VALUE(col)`: 取窗口帧内**最后一行**的值 ⚠️ 注意默认帧！
- `NTH_VALUE(col, n)`: 取窗口帧内**第 n 行**的值

**常见陷阱**: `LAST_VALUE` 在默认帧（RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW）下，
"最后一行" 是当前行，不是分区最后一行！
需要显式指定 `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`。

In [ ]:
# FIRST_VALUE / LAST_VALUE / NTH_VALUE 演示
result = con.execute("""
SELECT
    name,
    department,
    salary,
    -- 部门薪资最高者的姓名
    FIRST_VALUE(name) OVER (
        PARTITION BY department
        ORDER BY salary DESC
    ) AS highest_earner,
    -- 部门薪资最低者（需要完整帧）
    LAST_VALUE(name) OVER (
        PARTITION BY department
        ORDER BY salary DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING  -- 关键!
    ) AS lowest_earner,
    -- 部门薪资第二高者
    NTH_VALUE(name, 2) OVER (
        PARTITION BY department
        ORDER BY salary DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS second_highest_earner,
    -- 与最高薪资的差距
    FIRST_VALUE(salary) OVER (
        PARTITION BY department ORDER BY salary DESC
    ) - salary AS gap_from_top
FROM employees
ORDER BY department, salary DESC
""").df()

print("部门薪资参照值分析（注意 LAST_VALUE 需要指定完整帧）:")
result

---

## 6. NTILE 分桶

`NTILE(n)` 将窗口中的行**均匀分配到 n 个桶**中，返回桶编号（1 到 n）。

**典型应用**:
- 将用户按消费金额分为四分位数（Q1/Q2/Q3/Q4）
- 将商品按销量分为高/中/低销量段
- A/B 测试分组

**注意**: 如果行数不能被 n 整除，前几个桶会多一行。

In [ ]:
# NTILE 分桶演示：将员工按薪资分为四分位数
result = con.execute("""
SELECT
    name,
    department,
    salary,
    -- 全公司四分位数
    NTILE(4) OVER (ORDER BY salary) AS salary_quartile,
    -- 部门内四分位数
    NTILE(4) OVER (PARTITION BY department ORDER BY salary) AS dept_salary_quartile,
    -- 按薪资分为3个层次
    CASE NTILE(3) OVER (ORDER BY salary)
        WHEN 1 THEN 'Low'
        WHEN 2 THEN 'Mid'
        WHEN 3 THEN 'High'
    END AS salary_tier
FROM employees
ORDER BY salary DESC
""").df()

print("员工薪资分桶分析:")
result

In [ ]:
# 综合案例：完整的销售分析报表
result = con.execute("""
WITH sales_summary AS (
    SELECT
        salesperson,
        SUM(amount) AS total_sales,
        COUNT(*) AS num_transactions,
        AVG(amount) AS avg_transaction
    FROM sales
    GROUP BY salesperson
)
SELECT
    salesperson,
    ROUND(total_sales, 2)       AS total_sales,
    num_transactions,
    ROUND(avg_transaction, 2)   AS avg_transaction,
    -- 排名
    RANK() OVER (ORDER BY total_sales DESC)           AS sales_rank,
    -- 百分位
    ROUND(
        PERCENT_RANK() OVER (ORDER BY total_sales) * 100,
        1
    )                                                  AS percentile,
    -- 分桶（3档）
    CASE NTILE(3) OVER (ORDER BY total_sales)
        WHEN 1 THEN 'Bronze'
        WHEN 2 THEN 'Silver'
        WHEN 3 THEN 'Gold'
    END                                                AS tier,
    -- 与第一名差距
    ROUND(
        FIRST_VALUE(total_sales) OVER (ORDER BY total_sales DESC) - total_sales,
        2
    )                                                  AS gap_from_leader
FROM sales_summary
ORDER BY total_sales DESC
""").df()

print("综合销售员绩效分析报表:")
result

---

## 练习题

以下练习模拟真实的高级数据工程师面试场景，请独立完成后再查看提示。

In [ ]:
# 练习数据准备
con.execute("""
CREATE OR REPLACE TABLE orders AS
SELECT * FROM (VALUES
    (1001, 'C001', '2024-01-05', 250.00),
    (1002, 'C002', '2024-01-08', 180.00),
    (1003, 'C001', '2024-01-15', 320.00),
    (1004, 'C003', '2024-01-20', 95.00),
    (1005, 'C002', '2024-02-03', 450.00),
    (1006, 'C001', '2024-02-10', 120.00),
    (1007, 'C003', '2024-02-14', 680.00),
    (1008, 'C002', '2024-02-25', 210.00),
    (1009, 'C001', '2024-03-02', 540.00),
    (1010, 'C003', '2024-03-10', 370.00),
    (1011, 'C001', '2024-03-18', 890.00),
    (1012, 'C002', '2024-03-22', 155.00)
) t(order_id, customer_id, order_date, amount)
""")

con.execute("""
CREATE OR REPLACE TABLE stock_prices AS
SELECT * FROM (VALUES
    ('AAPL', DATE '2024-01-02', 185.50),
    ('AAPL', DATE '2024-01-03', 183.20),
    ('AAPL', DATE '2024-01-04', 182.10),
    ('AAPL', DATE '2024-01-05', 187.30),
    ('AAPL', DATE '2024-01-08', 188.90),
    ('AAPL', DATE '2024-01-09', 186.40),
    ('AAPL', DATE '2024-01-10', 190.20),
    ('GOOGL', DATE '2024-01-02', 140.30),
    ('GOOGL', DATE '2024-01-03', 141.50),
    ('GOOGL', DATE '2024-01-04', 139.80),
    ('GOOGL', DATE '2024-01-05', 143.20),
    ('GOOGL', DATE '2024-01-08', 145.60),
    ('GOOGL', DATE '2024-01-09', 144.30),
    ('GOOGL', DATE '2024-01-10', 147.90)
) t(symbol, trade_date, close_price)
""")

print("练习数据准备完成！")
print("\n--- orders 表 ---")
display(con.execute("SELECT * FROM orders ORDER BY customer_id, order_date").df())
print("\n--- stock_prices 表 ---")
display(con.execute("SELECT * FROM stock_prices ORDER BY symbol, trade_date").df())

### 练习 1: 客户订单序号与首次/最近购买

**需求**: 对每个客户的订单按时间排序，计算：
1. 每个订单在该客户历史中的序号（第几次购买）
2. 客户的首次购买金额
3. 客户的最近一次购买金额
4. 当前订单与上一次订单金额的差值

**预期输出列**: order_id, customer_id, order_date, amount, order_seq, first_order_amount, latest_order_amount, amount_diff_from_prev

In [ ]:
# 练习 1: 请填写 TODO 部分
result = con.execute("""
SELECT
    order_id,
    customer_id,
    order_date,
    amount,
    -- TODO 1: 每个客户的订单序号
    ROW_NUMBER() OVER (TODO) AS order_seq,
    -- TODO 2: 客户首次订单金额
    FIRST_VALUE(amount) OVER (TODO) AS first_order_amount,
    -- TODO 3: 客户最近订单金额（使用 LAST_VALUE，注意帧范围）
    LAST_VALUE(amount) OVER (TODO) AS latest_order_amount,
    -- TODO 4: 与上一次订单金额的差值
    amount - LAG(amount) OVER (TODO) AS amount_diff_from_prev
FROM orders
ORDER BY customer_id, order_date
""").df()
result

In [ ]:
# 练习 1 参考答案（运行查看）
result = con.execute("""
SELECT
    order_id,
    customer_id,
    order_date,
    amount,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS order_seq,
    FIRST_VALUE(amount) OVER (
        PARTITION BY customer_id ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS first_order_amount,
    LAST_VALUE(amount) OVER (
        PARTITION BY customer_id ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS latest_order_amount,
    amount - LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS amount_diff_from_prev
FROM orders
ORDER BY customer_id, order_date
""").df()
result

### 练习 2: 股票价格滑动分析

**需求**: 对股票价格数据计算：
1. 3日滑动平均收盘价
2. 每日涨跌幅（相对前一日的百分比变化）
3. 3日最高价（滑动最大值）
4. 每只股票当前价格相对其历史最高价的折扣百分比

**提示**: 使用 `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` 计算3日窗口

In [ ]:
# 练习 2: 请填写 TODO 部分
result = con.execute("""
SELECT
    symbol,
    trade_date,
    close_price,
    -- TODO 1: 3日滑动平均（当前行 + 前2行）
    ROUND(AVG(close_price) OVER (TODO), 2) AS ma_3day,
    -- TODO 2: 日涨跌幅百分比（与前一日对比）
    ROUND((close_price - LAG(close_price) OVER (TODO))
          / LAG(close_price) OVER (TODO) * 100, 2) AS daily_return_pct,
    -- TODO 3: 3日内最高价
    MAX(close_price) OVER (TODO) AS high_3day,
    -- TODO 4: 距历史最高价的折扣比例
    ROUND((MAX(close_price) OVER (TODO) - close_price)
          / MAX(close_price) OVER (TODO) * 100, 2) AS pct_below_all_time_high
FROM stock_prices
ORDER BY symbol, trade_date
""").df()
result

In [ ]:
# 练习 2 参考答案
result = con.execute("""
SELECT
    symbol,
    trade_date,
    close_price,
    ROUND(AVG(close_price) OVER (
        PARTITION BY symbol ORDER BY trade_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS ma_3day,
    ROUND((close_price - LAG(close_price) OVER (PARTITION BY symbol ORDER BY trade_date))
          / LAG(close_price) OVER (PARTITION BY symbol ORDER BY trade_date) * 100,
          2) AS daily_return_pct,
    MAX(close_price) OVER (
        PARTITION BY symbol ORDER BY trade_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS high_3day,
    ROUND((MAX(close_price) OVER (PARTITION BY symbol) - close_price)
          / MAX(close_price) OVER (PARTITION BY symbol) * 100,
          2) AS pct_below_all_time_high
FROM stock_prices
ORDER BY symbol, trade_date
""").df()
result

### 练习 3: 客户分层分析

**需求**: 按每个客户的总消费金额进行分层：
1. 使用 `NTILE(3)` 将客户分为 VIP / Regular / Occasional 三层
2. 计算每个客户在所有客户中的消费排名
3. 计算每个客户消费额占总消费额的百分比
4. 输出客户分层报告

In [ ]:
# 练习 3: 客户分层分析（填写 TODO）
result = con.execute("""
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(amount)   AS total_spent,
        COUNT(*)      AS num_orders,
        AVG(amount)   AS avg_order_value
    FROM orders
    GROUP BY customer_id
)
SELECT
    customer_id,
    ROUND(total_spent, 2)  AS total_spent,
    num_orders,
    -- TODO 1: 消费排名
    RANK() OVER (TODO) AS spending_rank,
    -- TODO 2: 消费百分位
    ROUND(PERCENT_RANK() OVER (TODO) * 100, 1) AS percentile,
    -- TODO 3: 占总消费额比例
    ROUND(total_spent / SUM(total_spent) OVER () * 100, 2) AS pct_of_total,
    -- TODO 4: 客户分层（NTILE 分3层，3=VIP, 2=Regular, 1=Occasional）
    CASE NTILE(3) OVER (TODO)
        WHEN 3 THEN 'VIP'
        WHEN 2 THEN 'Regular'
        WHEN 1 THEN 'Occasional'
    END AS customer_tier
FROM customer_totals
ORDER BY total_spent DESC
""").df()
result

In [ ]:
# 练习 3 参考答案
result = con.execute("""
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(amount)   AS total_spent,
        COUNT(*)      AS num_orders,
        AVG(amount)   AS avg_order_value
    FROM orders
    GROUP BY customer_id
)
SELECT
    customer_id,
    ROUND(total_spent, 2)  AS total_spent,
    num_orders,
    RANK() OVER (ORDER BY total_spent DESC) AS spending_rank,
    ROUND(PERCENT_RANK() OVER (ORDER BY total_spent) * 100, 1) AS percentile,
    ROUND(total_spent / SUM(total_spent) OVER () * 100, 2) AS pct_of_total,
    CASE NTILE(3) OVER (ORDER BY total_spent)
        WHEN 3 THEN 'VIP'
        WHEN 2 THEN 'Regular'
        WHEN 1 THEN 'Occasional'
    END AS customer_tier
FROM customer_totals
ORDER BY total_spent DESC
""").df()
result

### 练习 4: 连续增长检测（高难度）

**需求**: 找出每个客户中，连续两次购买金额都在增长的订单序列。
即：找出满足 `amount > prev_amount AND amount > prev_prev_amount` 的订单。

**提示**: 使用 `LAG(amount, 1)` 和 `LAG(amount, 2)` 获取前1次和前2次金额。

In [ ]:
# 练习 4: 连续增长检测（填写 TODO）
result = con.execute("""
WITH order_with_prev AS (
    SELECT
        order_id,
        customer_id,
        order_date,
        amount,
        -- TODO 1: 获取前一次购买金额
        LAG(amount, 1) OVER (TODO) AS prev_amount,
        -- TODO 2: 获取前两次购买金额
        LAG(amount, 2) OVER (TODO) AS prev_prev_amount
    FROM orders
)
SELECT
    order_id,
    customer_id,
    order_date,
    amount,
    prev_amount,
    prev_prev_amount,
    -- TODO 3: 标记是否连续增长
    CASE WHEN TODO THEN '连续增长' ELSE '非连续增长' END AS growth_status
FROM order_with_prev
WHERE prev_prev_amount IS NOT NULL  -- 只看有足够历史数据的记录
ORDER BY customer_id, order_date
""").df()
result

In [ ]:
# 练习 4 参考答案
result = con.execute("""
WITH order_with_prev AS (
    SELECT
        order_id,
        customer_id,
        order_date,
        amount,
        LAG(amount, 1) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_amount,
        LAG(amount, 2) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_prev_amount
    FROM orders
)
SELECT
    order_id,
    customer_id,
    order_date,
    amount,
    prev_amount,
    prev_prev_amount,
    CASE
        WHEN amount > prev_amount AND prev_amount > prev_prev_amount
        THEN '连续增长'
        ELSE '非连续增长'
    END AS growth_status
FROM order_with_prev
WHERE prev_prev_amount IS NOT NULL
ORDER BY customer_id, order_date
""").df()
result

### 练习 5: 薪资带宽分析（综合题）

**需求**: 对 employees 表进行完整的薪资带宽分析：
1. 每个部门的薪资中位数（使用 `PERCENTILE_CONT(0.5)`）
2. 每个员工的薪资在部门内的排名（使用 DENSE_RANK）
3. 每个员工薪资偏离部门均值的标准差倍数（Z-score）
4. 标记薪资异常值（Z-score 绝对值 > 1）

In [ ]:
# 练习 5: 薪资带宽分析（填写 TODO）
result = con.execute("""
SELECT
    name,
    department,
    salary,
    -- TODO 1: 部门内薪资排名
    DENSE_RANK() OVER (TODO) AS dept_rank,
    -- TODO 2: 部门平均薪资
    ROUND(AVG(salary) OVER (TODO), 2) AS dept_avg_salary,
    -- TODO 3: 部门薪资标准差
    ROUND(STDDEV(salary) OVER (TODO), 2) AS dept_salary_stddev,
    -- TODO 4: Z-score = (salary - dept_avg) / dept_stddev
    ROUND(
        (salary - AVG(salary) OVER (TODO))
        / NULLIF(STDDEV(salary) OVER (TODO), 0),
        2
    ) AS z_score,
    -- TODO 5: 标记异常值
    CASE
        WHEN ABS((salary - AVG(salary) OVER (TODO))
             / NULLIF(STDDEV(salary) OVER (TODO), 0)) > 1
        THEN 'Outlier'
        ELSE 'Normal'
    END AS salary_status
FROM employees
ORDER BY department, salary DESC
""").df()
result

In [ ]:
# 练习 5 参考答案
result = con.execute("""
SELECT
    name,
    department,
    salary,
    DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_rank,
    ROUND(AVG(salary) OVER (PARTITION BY department), 2) AS dept_avg_salary,
    ROUND(STDDEV(salary) OVER (PARTITION BY department), 2) AS dept_salary_stddev,
    ROUND(
        (salary - AVG(salary) OVER (PARTITION BY department))
        / NULLIF(STDDEV(salary) OVER (PARTITION BY department), 0),
        2
    ) AS z_score,
    CASE
        WHEN ABS(
            (salary - AVG(salary) OVER (PARTITION BY department))
            / NULLIF(STDDEV(salary) OVER (PARTITION BY department), 0)
        ) > 1 THEN 'Outlier'
        ELSE 'Normal'
    END AS salary_status
FROM employees
ORDER BY department, salary DESC
""").df()
result

---

## 复习要点

### 核心概念速查

| 场景 | 推荐函数 |
|------|----------|
| 每组取 Top N | `ROW_NUMBER() + CTE + WHERE rn <= N` |
| 并列排名（允许跳号）| `RANK()` |
| 并列排名（不允许跳号）| `DENSE_RANK()` |
| 环比 / 同比计算 | `LAG(col, 1)` / `LAG(col, 12)` |
| 累计运行总计 | `SUM() OVER (ORDER BY ...)` |
| N日滑动均值 | `AVG() OVER (ROWS BETWEEN N-1 PRECEDING AND CURRENT ROW)` |
| 分区内第一/最后值 | `FIRST_VALUE() / LAST_VALUE()` (注意帧范围) |
| 用户分层 | `NTILE(N)` |

### 常见陷阱

1. **LAST_VALUE 陷阱**: 默认帧只到当前行，需要加 `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`
2. **ROWS vs RANGE**: 有并列数据时两者结果不同，默认是 RANGE
3. **PARTITION BY 为空**: 不写 PARTITION BY 时整个结果集作为一个窗口
4. **NULL 值处理**: LAG/LEAD 在边界行返回 NULL，需用第三个参数指定默认值

### 面试高频问题

- "用 SQL 找出每个部门薪资最高的员工" → ROW_NUMBER + CTE
- "计算用户的次日留存率" → LAG/LEAD + 日期计算
- "计算每个用户的累计消费额" → SUM OVER (ORDER BY date)
- "找出连续增长的时间序列" → LAG(n) 多层对比
- "RANK 和 DENSE_RANK 的区别" → 必须背下并列处理逻辑